In [1]:
"""
Program for identifying stars in DECAM images.
"""

import numpy as np
from astropy.stats import sigma_clipped_stats
from astropy.table import Column
from astropy.wcs import WCS
import astropy.units as u
from astropy.coordinates import SkyCoord
from photutils.detection import DAOStarFinder

def cross_match(ra_new: np.array, dec_new: np.array, ra_catalog: np.array, dec_catalog: np.array):
    "Cross match to sets of coordinates."
    c_new = SkyCoord(ra = ra_new*u.deg, dec = dec_new*u.deg)
    c_catalog = SkyCoord(ra = ra_catalog*u.deg, dec = dec_catalog*u.deg)
    idx, d2d, _ = c_new.match_to_catalog_sky(c_catalog)
    max_separation = 1.0 * u.arcsec
    separation_constraint = d2d < max_separation
    c_idx = separation_constraint
    catalog_idx = idx[c_idx]
    return c_idx, catalog_idx


def find_stars_single(fits_hdu):
    "Finds stars in a fits data image and returns a catalog."
    _, median, std = sigma_clipped_stats(fits_hdu.data, sigma=3.0)
    daofind = DAOStarFinder(fwhm=5, threshold=20*std)
    sources = daofind(fits_hdu.data - median)

    #have to get RA and Dec because x, y are local positions
    x_pos = np.array(list(sources['xcentroid']))
    y_pos = list(sources['ycentroid'])
    local_wcs = WCS(fits_hdu.header)
    ras, decs = local_wcs.pixel_to_world_values(x_pos, y_pos)
    col_ra = Column(name='RA', data = ras)
    col_dec = Column(name='Dec', data = decs)
    sources.add_columns([col_ra, col_dec])

    return sources

/Users/aishwarya/.local/share/mamba/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
"""Module to handle sextracor inputs and outputs."""

from typing import List, Tuple
from pandas import DataFrame

def _get_column_names(read_line_object: List) -> List:
    """Reads the header info of the sextractor catalog."""
    header = [line.split()[2] for line in read_line_object if line.startswith('#') and len(line.split()) > 2]

    return header

def _get_rows(read_line_object: List) -> List:
    """Takes the readline object and reads in the rows."""
    data = [list(map(float, line.strip().split(','))) for line in read_line_object if line[0] != '#']
    return data

def split_names_and_data(read_line_object: List) -> Tuple[List, List]:
    """Takes the read in sextractor file and splits headers and columns."""
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> DataFrame:
    """Reads in the sextractor catalog."""
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    data_frame = DataFrame(data, columns = column_names)
    return data_frame


#if __name__ == '__main__':
   # INFILE = '/home/trystan/Desktop/Work/PhD/DECAM/correct_stacks/i/test.cat'
   # cat_df = read_cat(INFILE)

In [3]:
"""Main class for DECAM images"""

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import vstack
from reproject.mosaicking import find_optimal_celestial_wcs
from tqdm import tqdm
#from find_stars import find_stars_single, cross_match

def write_positions_as_region_file(positions: list, outfile: str, color: str):
    """Takes a list of positions and puts them in the ds9 region format."""
    with open(outfile, 'w', encoding="utf-8") as file:
        for position in positions:
            file.write(f'point({position[0]},{position[1]}) # point=circle 20  color={color}\n')

class DecamImage:
    """Loads in DECAM image and produces a star catalog."""
    def __init__(self, file_name: str):
        self.hdu_list = fits.open(file_name)
        self.main_wcs, _ = find_optimal_celestial_wcs(self.hdu_list[1:])
        self.create_star_catalog()

    def create_star_catalog(self):
        """Finding the stars across the entire stacked image."""
        star_catalogs = [find_stars_single(hdu) for hdu in tqdm(self.hdu_list[1:])]
        self.star_catalog = vstack(star_catalogs)

    def _convert_irsf_to_pixels(self, ra_array, dec_array):
        """Takes RA and Dec positions and puts them into pixel coordinates of the image."""
        x_pos, y_pos = self.main_wcs.world_to_pixel_values(ra_array, dec_array)
        positions = np.transpose((x_pos, y_pos))
        return positions

    def create_decam_cat_regions_file(self, outfile: str):
        """Create decam star-cat as a region file."""
        ras = np.array(list(self.star_catalog['RA']))
        decs = np.array(list(self.star_catalog['Dec']))
        self.create_region_file(ras, decs, outfile)
    
    def create_region_file(self, ra_array, dec_array, outfile: str, color='green'):
        """Generalized function to write region files given ra and dec arrays."""
        positions = self._convert_irsf_to_pixels(ra_array, dec_array)
        write_positions_as_region_file(positions, outfile, color=color)

    def create_cat_regions_file(self, outfile: str):
        """Takes a star catalog and converts it into a .reg file for ds9."""
        ras = np.array(list(self.star_catalog['RA']))
        decs = np.array(list(self.star_catalog['Dec']))
        x_pos, y_pos = self.main_wcs.world_to_pixel_values(ras, decs)
        positions = np.transpose((x_pos, y_pos))
        with open(outfile, 'w', encoding="utf-8") as file:
            for position in positions:
                file.write(f'point({position[0]},{position[1]}) # point=circle 20 \n')

    def cross_match_with_panstars(self, pan_stars_catalog_file_name: str):
        """panstars catalog must be in correct form from prep_panstars.py"""
        panstars_catalog = pd.read_csv(pan_stars_catalog_file_name)
        ra_panstars = np.array(list(panstars_catalog['raMean']))
        dec_panstars = np.array(list(panstars_catalog['decMean']))
        idx_decam, idx_panstars = cross_match(self.star_catalog['RA'], self.star_catalog['Dec'],
                                              ra_panstars, dec_panstars)
        matched_panstars_catalog = panstars_catalog.iloc[idx_panstars]
        matched_decam_catalog = self.star_catalog[idx_decam]

        self.create_region_file(matched_panstars_catalog['raMean'],
                                matched_panstars_catalog['decMean'],
                                'panstars_matches.reg',
                                color = 'red')

        self.create_region_file(matched_decam_catalog['RA'],
                                matched_decam_catalog['Dec'], 'decam_matches.reg')
        return matched_decam_catalog, matched_panstars_catalog


#if __name__ == '__main__':
  #  INFILE = '../correct_stacks/i/c4d_211021_003940_osj_i_vik1_skysubtracted.fits.fz'
   # PAN_CAT = '../PANSTARS/PANSTARS_i.csv'
    #test = DecamImage(INFILE)
    #test.create_decam_cat_regions_file('test.reg')
   # thing = test.cross_match_with_panstars(PAN_CAT)

In [9]:
"""Main class for sextractor catalogs."""

import numpy as np
import pylab as plt
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd




class SExtractorCat:
    """Main class for sextractor catalogs."""
    def __init__(self, sextractor_file_name: str):
        """Initializing class"""
        self.catalog = read_cat(sextractor_file_name)

    def cross_match_with_panstars(self, pan_stars_file_name:str):
        """reducing the sextractor catalog to only those which are in panstars."""
        panstars_catalog = pd.read_csv(pan_stars_file_name)
        ra_panstars = np.array(list(panstars_catalog['raMean']))
        dec_panstars = np.array(list(panstars_catalog['decMean']))

        ra_decam = np.array(list(self.catalog['ALPHAPEAK_J2000']))
        dec_decam = np.array(list(self.catalog['DELTAPEAK_J2000']))

        idx_decam, idx_panstars = cross_match(ra_decam, dec_decam, ra_panstars, dec_panstars)

        matched_panstars_catalog = panstars_catalog.iloc[idx_panstars]
        matched_decam_catalog = self.catalog[idx_decam]
        return matched_decam_catalog, matched_panstars_catalog

    def quick_look(self) -> None:
        """Plots the ra and dec scatter plot to make sure things look ok."""
        ra_decam = np.array(list(self.catalog['ALPHAPEAK_J2000']))
        dec_decam = np.array(list(self.catalog['DELTAPEAK_J2000']))
        plt.scatter(ra_decam, dec_decam)
        plt.show()

    def to_region_file(self, decam_file: str, outfile: str, color: str = 'orange', only_galaxies = False) -> None:
        """Converts the points into a region file."""
        hdu = fits.open(decam_file)
        wcs, _ = find_optimal_celestial_wcs(hdu[1:])

        ra_positions = self.catalog['ALPHAPEAK_J2000'].values
        dec_positions = self.catalog['DELTAPEAK_J2000'].values
        x_positions, y_positions = wcs.world_to_pixel_values(ra_positions, dec_positions)

        if only_galaxies is True:
            cut = np.where(self.catalog['CLASS_STAR'].values < 0.5)
            x_positions = x_positions[cut]
            y_positions = y_positions[cut]
        write_positions_as_region_file(np.transpose((x_positions, y_positions)), outfile, color)

    def remove_sources_based_on_exposure_map(self, exp_map: str) -> None:
        """Uses an expsore map to exlude sources with no exposure times."""
        hdul = fits.open(exp_map)
        exptime = np.zeros(len(self.catalog['ALPHAPEAK_J2000']))
        for hdu in hdul[1:]:
            wcs = WCS(hdu.header)
            x_pixels, y_pixels = wcs.world_to_pixel_values(
                self.catalog['ALPHAPEAK_J2000'], self.catalog['DELTAPEAK_J2000'])
            x_pixels, y_pixels = x_pixels.astype(int), y_pixels.astype(int)
            cut_y = np.where((y_pixels < hdu.shape[0]) & (y_pixels >= 0))[0]
            cut_x = np.where((x_pixels < hdu.shape[1]) & (x_pixels >= 0))[0]
            final_cut = np.intersect1d(cut_x, cut_y).astype(int)
            vals = [hdu.data[y_pixels[index], x_pixels[index]] for index in final_cut]
            exptime[final_cut] = vals
        self.catalog['exptime'] = exptime
        self.catalog = self.catalog[self.catalog.exptime != 0]

In [10]:


"""Plotting package to make all the plots look like publish ready plots."""

import matplotlib as mpl
import pylab as plt

def apply_global_settings() -> None:
    """Global settings."""
    mpl.rcParams.update({'font.size': 2})
    #mpl.rcParams['font.family'] = 'Avenir'
    plt.rcParams['font.size'] = 8
    plt.rcParams['axes.linewidth'] = 1.1
    mpl.rc('xtick', labelsize=8)
    mpl.rc('ytick', labelsize=8)

def prettify_plot(x_label: str, y_label: str) -> None:
    """Makes the plots look good."""
    plt.xlabel(x_label,fontsize=12)
    plt.ylabel(y_label,fontsize=12)
    plt.minorticks_on()
    plt.tick_params(which='both', width=1.2,direction='in')
    plt.tick_params(which='major', length=3, direction='in')

def start_plot(x_label: str, y_label: str) -> plt.Figure:
    """Starting the plot."""
    fig = plt.figure(figsize = (3.54, 3.54), dpi = 600)
    prettify_plot(x_label, y_label)
    return fig

def prettify_axis(axis: plt.axes, x_label: str, y_label: str) -> None:
    """Does prettify plot but for a given axis"""
    axis.set_xlabel(x_label, fontsize=12)
    axis.set_ylabel(y_label, fontsize=12)
    axis.minorticks_on()
    axis.tick_params(which='both', width=1.2,direction='in')
    axis.tick_params(which='major', length=3, direction='in')


def end_plot(outfile: str) -> None:
    """Saves the figure correctly."""
    plt.savefig(outfile, bbox_inches = 'tight', transparent = False)

apply_global_settings()

In [11]:
"""
Calculate the zero points of the i-band and z-band
"""

import os
from typing import Tuple
import numpy as np
import pylab as plt
from scipy.optimize import curve_fit
import pandas as pd
from astropy.stats import sigma_clipped_stats






def remove_outliers(array, sigma):
    """returns a mask of values within the given sigma."""
    median, std = np.median(array), np.std(array)
    good_idx = np.where((array >= median-sigma*std) & (array <= median + sigma*std))[0]
    return good_idx

def read_in_wide_band(
        sextractor_file_name: str, panstars_cat_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reading in the catalogs."""
    decam_catalog = SExtractorCat(sextractor_file_name)
    decam_catalog, pan_cat = decam_catalog.cross_match_with_panstars(panstars_cat_name)
    return decam_catalog, pan_cat

def get_broad_band_mags(decam_df: pd.DataFrame, panstars_cat: pd.DataFrame, band: str) -> Tuple:
    """Gets the magnitudes and errors for a wide-band filter."""

    dec_mags = decam_df['MAG_APER'].values
    dec_mags_uncertainty = decam_df['MAGERR_APER'].values
    pan_mags = panstars_cat[f'{band}MeanPSFMag'].values
    pan_mags_uncertainty = panstars_cat[f'{band}MeanPSFMagErr'].values

    converted_mags, converted_mags_errors = CONVERT[band](panstars_cat)

    return dec_mags, dec_mags_uncertainty, converted_mags,\
          converted_mags_errors, pan_mags, pan_mags_uncertainty


def convert_panstars_i_dec_mags(panstars_cat: pd.DataFrame):
    """Converts the panstars magnitudes into decam magnitudes.
    see: https://des.ncsa.illinois.edu/releases/dr2/dr2-docs/dr2-transformations"""
    r_panstars_mags = panstars_cat['rMeanPSFMag'].values
    i_panstars_mags = panstars_cat['iMeanPSFMag'].values
    i_uncertainties = panstars_cat['iMeanPSFMagErr'].values
    r_uncertainties = panstars_cat['rMeanPSFMagErr'].values

    i_decam = i_panstars_mags - 0.155 * (r_panstars_mags - i_panstars_mags) + 0.015
    converted_i_uncertainties = np.hypot(
        i_uncertainties, 0.155*np.hypot(i_uncertainties, r_uncertainties))

    return i_decam, converted_i_uncertainties


def convert_panstars_z_dec_mags(panstars_cat: pd.DataFrame):
    """Converts the panstars magnitudes into decam magnitudes.
    see: https://des.ncsa.illinois.edu/releases/dr2/dr2-docs/dr2-transformations"""
    r_panstars_mags = panstars_cat['rMeanPSFMag'].values
    i_panstars_mags = panstars_cat['iMeanPSFMag'].values
    z_panstars_mags = panstars_cat['zMeanPSFMag'].values
    i_uncertainties = panstars_cat['iMeanPSFMagErr'].values
    r_uncertainties = panstars_cat['rMeanPSFMagErr'].values
    z_uncertainties = panstars_cat['zMeanPSFMagErr'].values

    z_decam = z_panstars_mags - 0.114 * (r_panstars_mags - i_panstars_mags) - 0.010
    converted_z_uncertainties = np.hypot(
        z_uncertainties, 0.114*np.hypot(i_uncertainties, r_uncertainties))

    return z_decam, converted_z_uncertainties


CONVERT = {
    'i': convert_panstars_i_dec_mags,
    'z': convert_panstars_z_dec_mags,
}

def prepare_plotting_data(sextractor_file_name: str, panstars_file_name: str, band: str):
    """Gets all the necessary variables for plotting ready and removes outliers."""
    decam_catalog, panstars_catalog = read_in_wide_band(sextractor_file_name, panstars_file_name)
    mags = get_broad_band_mags(decam_catalog, panstars_catalog, band=band)
    msk = remove_outliers(mags[-2], sigma=2)
    cleaned_mags = [mag[msk] for mag in mags]
    return cleaned_mags


def straight_line(x_array, intercept):
    """y=mx +c with m=1. Line for fitting to the zpt plot."""
    return x_array + intercept

class BroadBand:
    """
    Broad Band class for sorting the different magnitudes
    and determine the zpt.
    """
    def __init__(self, panstars_cat_name: str, sextractor_cat_name: str, broadband: str):
        if broadband not in 'iz':
            raise ValueError('broadband needs to be either "i" or "z".')
        self.broadband = broadband
        self.sextractor_cat_name = sextractor_cat_name
        mags = prepare_plotting_data(sextractor_cat_name, panstars_cat_name, broadband)
        diffs = mags[2] - mags[0]
        diff_cut = np.where((diffs>30) & (diffs <32))[0]  # specifically for this research.
        good_values = np.where(mags[2] < 100)[0] # Obviously not physical if this isn't met.
        other_good_values = np.where(mags[0] < -6)[0] # specific for decam
        more_good_values = np.where(mags[3] < 1.)[0] # not physical to have errors more than 100 mags
        even_more = np.where(mags[1] < 1.)[0]
        good_values = np.intersect1d(good_values, other_good_values)
        good_values = np.intersect1d(good_values, more_good_values)
        good_values = np.intersect1d(good_values, diff_cut)
        good_values = np.intersect1d(good_values, even_more)
        self.measured_mags = mags[0][good_values]
        self.measured_mags_err = mags[1][good_values]
        self.expected_mags = mags[2][good_values]
        self.expected_mags_err = mags[3][good_values]

    def plot_zpt(self):
        """Plots the measured mags vs the expected mags."""
        start_plot('DECam mag', 'Expected mag - DECam mag')
        y_err = np.hypot(self.measured_mags_err, self.expected_mags_err)
        good = np.where(y_err < 1e5)[0]
        plt.errorbar(
            self.measured_mags[good], self.expected_mags[good] - self.measured_mags[good],
            xerr=self.measured_mags_err[good], yerr=y_err[good],
            fmt='ko', alpha=0.3, ms=2.5, elinewidth=1.5)
        x_fit, fit, fit_up, fit_down = self.fit_horizontal_line()
        plt.plot(x_fit, fit, ls='--', color='r', lw=3, zorder=1)
        plt.fill_between(x_fit, fit_up, fit_down, alpha=.5, color='r')
        end_plot(f'plots/{self.broadband}_zpt.png')
        plt.show()

    def fit_straight_line(self) -> Tuple:
        """Fitting y=x+c line to the data to determine c"""
        zpt, zpt_err = self.zero_point
        nstd = 5.
        popt_up = zpt + nstd * zpt_err
        popt_dw = zpt - nstd * zpt_err
        x_fit = np.linspace(np.sort(self.measured_mags)[0], np.sort(self.measured_mags)[-1])
        fit = straight_line(x_fit, zpt)
        fit_up = straight_line(x_fit, popt_up)
        fit_dw= straight_line(x_fit, popt_dw)
        return x_fit, fit, fit_up, fit_dw

    def fit_horizontal_line(self) ->Tuple:
        """Fitting y = c line."""
        zpt, zpt_err = self.zero_point[0], self.zero_point[1]
        nstd = 5.
        popt_up = zpt + nstd * zpt_err
        popt_dw = zpt - nstd * zpt_err
        x_fit = np.linspace(np.sort(self.measured_mags)[0], np.sort(self.measured_mags)[-1])
        fit = zpt * np.ones(len(x_fit))
        fit_up = popt_up * np.ones(len(x_fit))
        fit_down = popt_dw * np.ones(len(x_fit))
        return x_fit, fit, fit_up, fit_down

    @property
    def zero_point(self) -> Tuple[float, float]:
        """
        Determines the zero point by fitting a straight line and determining the intercept.
        """
        cut = np.where((self.measured_mags > -14) & (self.measured_mags < -11))
        a_fit, cov = curve_fit(
            straight_line, self.measured_mags[cut], self.expected_mags[cut],
            sigma=self.expected_mags_err[cut], absolute_sigma=True)
        uncertainties = np.sqrt(np.diag(cov))
        return a_fit[0], uncertainties[0]

    def determine_zero_point_prime(self, aperture_radius: float, seeing: float) -> float:
        """
        This is the zero point minus the k_correction. This means that the magnitudes 
        in the future would be determined by adding this constant to the k_correction
        which is dependent on the radius.

        Must provide the radius of the apertures used by sextractor to determine the magnitudes
        as well as the seeing of the image. Both must be in the same units.
        """
        return self.zero_point[0] - calculate_k_constant_mag(aperture_radius, seeing)


if __name__ == '__main__':
    INFILE_SEX_I = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_depth.cat'
    INFILE_PAN_I = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_panstarrs.csv'
    INFILE_SEX_Z = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_depth.cat'
    INFILE_PAN_Z = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_panstarrs.csv'

    INFILE_SEX_I_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/z_band_depth_decam_matched.cat'
    INFILE_PAN_I_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_panstarrs.csv'
    INFILE_SEX_Z_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/i_band_depth_decam_matched.cat'
    INFILE_PAN_Z_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_panstarrs.csv'

    SEEING_I = 1.17 # These comes from the seeing calculator.
    SEEING_Z = 1.23
    APERTURE_RADII = 1 #min kron radius

    SEEING_I_CDFS = 1.22
    SEEING_Z_CDFS = 1.14
    APERTURE_RADII_CDFS = 1 #min kron radius


    i_band_cdfs = BroadBand(INFILE_PAN_I_CDFS, INFILE_SEX_I_CDFS, 'i')
    print('i prime cdfs is: ', i_band_cdfs.determine_zero_point_prime(APERTURE_RADII_CDFS, SEEING_I_CDFS))
    i_band_cdfs.plot_zpt()
    os.system('mv plots/i_zpt.png plots/i_zpt_cdfs.png')

    z_band_cdfs = BroadBand(INFILE_PAN_Z_CDFS, INFILE_SEX_Z_CDFS, 'z')
    print('z prime cdfs is: ', z_band_cdfs.determine_zero_point_prime(APERTURE_RADII_CDFS, SEEING_Z_CDFS))
    z_band_cdfs.plot_zpt()
    os.system('mv plots/z_zpt.png plots/z_zpt_cdfs.png')

    i_band = BroadBand(INFILE_PAN_I, INFILE_SEX_I, 'i')
    print('i prime is: ', i_band.determine_zero_point_prime(APERTURE_RADII, SEEING_I))
    i_band.plot_zpt()

    z_band = BroadBand(INFILE_PAN_Z, INFILE_SEX_Z, 'z')
    print('z prime is: ', z_band.determine_zero_point_prime(APERTURE_RADII, SEEING_Z))
    z_band.plot_zpt()

ValueError: 0 columns passed, passed data had 16 columns

In [17]:
import os
from typing import Tuple
import numpy as np
import pandas as pd
import pylab as plt
from scipy.optimize import curve_fit
from astropy.stats import sigma_clipped_stats

# ----------------- Utility Functions -----------------
def remove_outliers(array, sigma):
    median, std = np.median(array), np.std(array)
    good_idx = np.where((array >= median - sigma * std) & (array <= median + sigma * std))[0]
    return good_idx

def straight_line(x_array, intercept):
    return x_array + intercept

# ----------------- Conversion Functions -----------------
def convert_panstars_i_dec_mags(panstars_cat: pd.DataFrame):
    r = panstars_cat['rMeanPSFMag'].values
    i = panstars_cat['iMeanPSFMag'].values
    r_err = panstars_cat['rMeanPSFMagErr'].values
    i_err = panstars_cat['iMeanPSFMagErr'].values

    i_decam = i - 0.155 * (r - i) + 0.015
    i_decam_err = np.hypot(i_err, 0.155 * np.hypot(i_err, r_err))
    return i_decam, i_decam_err

def convert_panstars_z_dec_mags(panstars_cat: pd.DataFrame):
    r = panstars_cat['rMeanPSFMag'].values
    i = panstars_cat['iMeanPSFMag'].values
    z = panstars_cat['zMeanPSFMag'].values
    r_err = panstars_cat['rMeanPSFMagErr'].values
    i_err = panstars_cat['iMeanPSFMagErr'].values
    z_err = panstars_cat['zMeanPSFMagErr'].values

    z_decam = z - 0.114 * (r - i) - 0.010
    z_decam_err = np.hypot(z_err, 0.114 * np.hypot(i_err, r_err))
    return z_decam, z_decam_err

CONVERT = {
    'i': convert_panstars_i_dec_mags,
    'z': convert_panstars_z_dec_mags,
}

# ----------------- SExtractor Wrapper -----------------
class SExtractorCat(pd.DataFrame):
    def __init__(self, file_path):
        column_names = [
            'NUMBER', 'ID_PARENT', 'FLUX_AUTO', 'FLUXERR_AUTO', 'MAG_AUTO', 'MAGERR_AUTO',
            'ALPHAPEAK_SKY', 'DELTAPEAK_SKY', 'ALPHAPEAK_J2000', 'DELTAPEAK_J2000',
            'X_IMAGE', 'Y_IMAGE', 'FLAGS', 'CLASS_STAR', 'RA_CORRECTED', 'DEC_CORRECTED'
        ]
        data = pd.read_csv(file_path, sep=',', skiprows=1, names=column_names)
        data['RA_CORRECTED'] = pd.to_numeric(data['RA_CORRECTED'], errors='coerce')
        data['DEC_CORRECTED'] = pd.to_numeric(data['DEC_CORRECTED'], errors='coerce')
        super().__init__(data)

        self['RA'] = self['RA_CORRECTED']
        self['DEC'] = self['DEC_CORRECTED']


    def cross_match_with_panstars(self, panstars_file):
        from astropy.coordinates import SkyCoord
        from astropy import units as u
        pan = pd.read_csv(panstars_file)
        c1 = SkyCoord(ra=self['RA'] * u.deg, dec=self['DEC'] * u.deg)
        c2 = SkyCoord(ra=pan['ra'] * u.deg, dec=pan['dec'] * u.deg)
        idx, d2d, _ = c1.match_to_catalog_sky(c2)
        sep_constraint = d2d < 1.0 * u.arcsec
        matched_dec = self[sep_constraint].reset_index(drop=True)
        matched_pan = pan.iloc[idx[sep_constraint]].reset_index(drop=True)
        return matched_dec, matched_pan

# ----------------- Main BroadBand Class -----------------
class BroadBand:
    def __init__(self, panstars_cat_name: str, sextractor_cat_name: str, broadband: str):
        if broadband not in 'iz':
            raise ValueError('broadband needs to be either "i" or "z".')
        self.broadband = broadband

        decam_catalog = SExtractorCat(sextractor_cat_name)
        decam_catalog, pan_cat = decam_catalog.cross_match_with_panstars(panstars_cat_name)

        dec_mags = decam_catalog['MAG_APER'].values
        dec_mags_err = decam_catalog['MAGERR_APER'].values
        converted_mags, converted_mags_err = CONVERT[broadband](pan_cat)

        diffs = converted_mags - dec_mags
        diff_cut = np.where((diffs > 30) & (diffs < 32))[0]
        good_values = np.where((converted_mags < 100) & (dec_mags < -6) & (converted_mags_err < 1) & (dec_mags_err < 1))[0]
        good_values = np.intersect1d(good_values, diff_cut)

        self.measured_mags = dec_mags[good_values]
        self.measured_mags_err = dec_mags_err[good_values]
        self.expected_mags = converted_mags[good_values]
        self.expected_mags_err = converted_mags_err[good_values]

    @property
    def zero_point(self):
        cut = np.where((self.measured_mags > -14) & (self.measured_mags < -11))
        a_fit, cov = curve_fit(
            straight_line, self.measured_mags[cut], self.expected_mags[cut],
            sigma=self.expected_mags_err[cut], absolute_sigma=True)
        return a_fit[0], np.sqrt(np.diag(cov))[0]

    def determine_zero_point_prime(self, aperture_radius: float, seeing: float) -> float:
        return self.zero_point[0] - calculate_k_constant_mag(aperture_radius, seeing)

    def plot_zpt(self):
        y_err = np.hypot(self.measured_mags_err, self.expected_mags_err)
        good = np.where(y_err < 1e5)[0]
        plt.errorbar(
            self.measured_mags[good], self.expected_mags[good] - self.measured_mags[good],
            xerr=self.measured_mags_err[good], yerr=y_err[good],
            fmt='ko', alpha=0.3, ms=2.5, elinewidth=1.5)
        x_fit, fit, fit_up, fit_down = self.fit_horizontal_line()
        plt.plot(x_fit, fit, ls='--', color='r', lw=3, zorder=1)
        plt.fill_between(x_fit, fit_up, fit_down, alpha=.5, color='r')
        plt.xlabel('DECam mag')
        plt.ylabel('Expected mag - DECam mag')
        plt.grid()
        plt.savefig(f'plots/{self.broadband}_zpt.png')
        plt.show()

    def fit_horizontal_line(self):
        zpt, zpt_err = self.zero_point
        nstd = 5.
        x_fit = np.linspace(min(self.measured_mags), max(self.measured_mags), 100)
        fit = zpt * np.ones_like(x_fit)
        fit_up = (zpt + nstd * zpt_err) * np.ones_like(x_fit)
        fit_down = (zpt - nstd * zpt_err) * np.ones_like(x_fit)
        return x_fit, fit, fit_up, fit_down

# ----------------- k correction function -----------------
def calculate_k_constant_mag(aperture_radius: float, seeing: float) -> float:
    # Example placeholder formula; replace with empirical or PSF-derived function
    return 2.5 * np.log10(1 - np.exp(-aperture_radius**2 / (2 * (seeing/2.355)**2)))

# ----------------- Main Script -----------------
if __name__ == '__main__':
    os.makedirs('plots', exist_ok=True)

    INFILE_SEX_I = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_depth.cat'
    INFILE_PAN_I = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_panstarrs.csv'
    INFILE_SEX_Z = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_depth.cat'
    INFILE_PAN_Z = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_panstarrs.csv'

    INFILE_SEX_I_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/z_band_depth_decam_matched.cat'
    INFILE_PAN_I_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band_panstarrs.csv'
    INFILE_SEX_Z_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/i_band_depth_decam_matched.cat'
    INFILE_PAN_Z_CDFS = '/Users/aishwarya/Documents/Lyman_alpha/CAT/z_band_panstarrs.csv'

    SEEING_I = 1.17
    SEEING_Z = 1.23
    APERTURE_RADII = 1

    SEEING_I_CDFS = 1.22
    SEEING_Z_CDFS = 1.14
    APERTURE_RADII_CDFS = 1

    i_band_cdfs = BroadBand(INFILE_PAN_I_CDFS, INFILE_SEX_I_CDFS, 'i')
    print('i prime cdfs is: ', i_band_cdfs.determine_zero_point_prime(APERTURE_RADII_CDFS, SEEING_I_CDFS))
    i_band_cdfs.plot_zpt()
    os.rename('plots/i_zpt.png', 'plots/i_zpt_cdfs.png')

    z_band_cdfs = BroadBand(INFILE_PAN_Z_CDFS, INFILE_SEX_Z_CDFS, 'z')
    print('z prime cdfs is: ', z_band_cdfs.determine_zero_point_prime(APERTURE_RADII_CDFS, SEEING_Z_CDFS))
    z_band_cdfs.plot_zpt()
    os.rename('plots/z_zpt.png', 'plots/z_zpt_cdfs.png')

    i_band = BroadBand(INFILE_PAN_I, INFILE_SEX_I, 'i')
    print('i prime is: ', i_band.determine_zero_point_prime(APERTURE_RADII, SEEING_I))
    i_band.plot_zpt()

    z_band = BroadBand(INFILE_PAN_Z, INFILE_SEX_Z, 'z')
    print('z prime is: ', z_band.determine_zero_point_prime(APERTURE_RADII, SEEING_Z))
    z_band.plot_zpt()


UnitTypeError: Longitude instances require units equivalent to 'rad', but no unit was given.

In [15]:
df = pd.read_csv('/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/z_band_depth_decam_matched.cat', delim_whitespace=True, comment='#')
print(df.columns)


Index(['2750.0,2852.0,913.9396,32.5879,12.5977,0.0387,357.555736,-31.8002253,357.555736,-31.8002253,8603.582,234.2746,0.0,0.0,357.55574323,-31.80023522'], dtype='object')


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_39345/47201594.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv('/Users/aishwarya/Documents/Lyman_alpha/CAT/cat/m/z_band_depth_decam_matched.cat', delim_whitespace=True, comment='#')


In [18]:
print(self[['RA', 'DEC']].dtypes)
print(self[['RA', 'DEC']].head())
print(self[['RA', 'DEC']].isna().sum())


NameError: name 'self' is not defined